# FARUKH SUPPLY — тест PP-StructureV3
Бесплатная отдельная проверка OCR, не обновление приложения. В меню «Среда выполнения» выберите CPU и запустите ячейки по очереди. Временная среда Google Colab, лимиты бесплатного тарифа возможны. Загружайте только документы, которые разрешено обрабатывать в облаке. Не публикуйте результаты и не сохраняйте ноутбук с выводом в общий доступ.

In [ ]:
%pip install -q 'paddlepaddle==3.2.2' 'paddleocr[doc-parser]==3.3.1'
print('Установка завершена; при запросе Colab перезапустите среду и переходите к следующей ячейке.')

In [ ]:
import os,time
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='True'
os.environ['OMP_NUM_THREADS']='4'
from paddleocr import PPStructureV3
from google.colab import files
from IPython.display import display, HTML
from html.parser import HTMLParser
from pathlib import Path
import pandas as pd, json, tempfile, zipfile, html
start=time.monotonic()
pipe=PPStructureV3(device='cpu',enable_mkldnn=False,cpu_threads=4,text_detection_model_name='PP-OCRv5_mobile_det',text_recognition_model_name='cyrillic_PP-OCRv5_mobile_rec',use_doc_orientation_classify=True,use_doc_unwarping=False,use_textline_orientation=True,use_formula_recognition=False,use_chart_recognition=False,use_seal_recognition=False,use_table_recognition=True)
print('Движок готов. Загрузка моделей:',round(time.monotonic()-start,1),'сек.')

## Загрузить накладные
Нажмите ▶ у следующей ячейки и выберите ровные фотографии. Для новой проверки повторите только эту ячейку. Результаты — распознанные таблицы без автоматических исправлений; обязательно сверяйте числа с оригиналом.

In [ ]:
class Cells(HTMLParser):
    def __init__(self):
        super().__init__();self.rows=[];self.row=None;self.cell=None
    def handle_starttag(self,tag,attrs):
        if tag=='tr':self.row=[]
        if tag in ('td','th'):self.cell=[]
    def handle_data(self,data):
        if self.cell is not None:self.cell.append(data)
    def handle_endtag(self,tag):
        if tag in ('td','th') and self.cell is not None:
            if self.row is not None:self.row.append(' '.join(''.join(self.cell).split()))
            self.cell=None
        if tag=='tr' and self.row is not None:self.rows.append(self.row);self.row=None

with tempfile.TemporaryDirectory() as tmp:
    root=Path(tmp)
    previous=os.getcwd()
    os.chdir(root)
    try:
        uploaded=files.upload()
    finally:
        os.chdir(previous)
    report=[]
    for i,(name,data) in enumerate(uploaded.items()):
        ext=Path(name).suffix.lower()
        if ext not in ('.jpg','.jpeg','.png','.webp','.pdf'):
            print('Неподдерживаемый файл:',name);continue
        path=root/(str(i)+ext);path.write_bytes(data)
        print('Документ:',name)
        t=time.monotonic()
        try:
            for p,r in enumerate(pipe.predict(input=str(path),use_chart_recognition=False),1):
                raw=r.json
                if callable(raw):raw=raw()
                if isinstance(raw,str):raw=json.loads(raw)
                raw=raw.get('res',raw)
                text=raw.get('overall_ocr_res',{}).get('rec_texts',[])
                tables=[]
                for table in raw.get('table_res_list',[]):
                    parser=Cells();parser.feed(table.get('pred_html',''))
                    if parser.rows:
                        tables.append(parser.rows)
                        display(pd.DataFrame(parser.rows))
                print('Страница',p,'· таблиц:',len(tables))
                print('Текст:', '\n'.join(text)[:3000])
                report.append({'file':name,'page':p,'tables':tables,'text':text,'seconds_elapsed':round(time.monotonic()-t,2)})
        except Exception as e:
            print('Ошибка:',str(e));report.append({'file':name,'error':str(e)})
        print('Время:',round(time.monotonic()-t,2),'сек.')
    result=Path(previous)/'FARUKH-OCR-test-result.json'
    result.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    if report:files.download(str(result))
    del uploaded
print('Исходные фото удалены из временной папки. Отчёт содержит распознанный текст: не публикуйте его.')